# Automotive Price Prediction — Auto Trader UK
### Advanced Machine Learning (AML) — MSc Data Science, Manchester Metropolitan University

**Task:** Predict UK vehicle listing prices from Auto Trader data using a full ML pipeline.

**Models evaluated:** Linear Regression · Random Forest · XGBoost · Ensemble Voting  
**Best result:** R² = 0.871 (Random Forest, no PCA)  
**Key insight:** SHAP analysis confirms vehicle make, model, and age as the dominant price drivers — mileage matters less than age for premium brands.

> **Prerequisite:** This notebook uses `cleaned_data.csv` produced by `vehicle_price_eda.ipynb`.

---
## Notebook Structure
1. Setup & Data Loading
2. Feature Engineering
3. Preprocessing Pipeline
4. Feature Selection — RFECV
5. PCA Exploration
6. Model Training & Evaluation
   - Linear Regression (baseline)
   - Random Forest
   - XGBoost + Hyperparameter Tuning
   - Ensemble Voting
7. Model Comparison
8. SHAP Feature Importance
9. Partial Dependence Plots

## 1. Setup & Data Loading

In [ ]:
!pip install -q category_encoders xgboost shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      cross_validate, GridSearchCV)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               VotingRegressor)
from sklearn.preprocessing import (PolynomialFeatures, OneHotEncoder, MinMaxScaler)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV, SelectKBest, f_regression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.inspection import PartialDependenceDisplay
import category_encoders as ce
import xgboost as xgb
import shap
shap.initjs()

print("All libraries imported successfully.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = 'cleaned_data.csv'   # Output from vehicle_price_eda.ipynb
TEST_SIZE   = 0.25
RANDOM_SEED = 42
TARGET_COL  = 'price'

NUMERICAL_FEATURES   = ['mileage', 'Age']
CATEGORICAL_FEATURES = ['standard_make', 'standard_model', 'standard_colour', 'body_type']
ONE_HOT_FEATURE      = ['fuel_type']

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Feature Engineering

Two polynomial interaction features are added to capture non-linear relationships
between mileage and vehicle age — the two strongest continuous price predictors:

- `poly_mile` — mileage (passthrough)
- `poly_Age`  — age (passthrough)
- `poly_mile_Age` — interaction term: mileage × age (captures combined depreciation effect)

In [ ]:
def add_polynomial_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add degree-2 interaction features for mileage and Age.
    Captures the combined depreciation effect of high mileage + old age.
    """
    df = df.copy()
    poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
    poly_arr = poly.fit_transform(df[['mileage', 'Age']])
    poly_df  = pd.DataFrame(
        poly_arr,
        columns=['poly_mile', 'poly_Age', 'poly_mile_Age'],
        index=df.index
    )
    return pd.concat([df, poly_df], axis=1)

df = add_polynomial_features(df)

# Separate target and features
target   = df[TARGET_COL]
features = df.drop(columns=[TARGET_COL])

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 3. Preprocessing Pipeline

A single `ColumnTransformer` handles all feature types consistently:

| Feature type | Columns | Transformation |
|---|---|---|
| Numerical | `mileage`, `Age` | Mean imputation → MinMaxScaler |
| High-cardinality categorical | `standard_make`, `standard_model`, `standard_colour`, `body_type` | Target encoding |
| Low-cardinality categorical | `fuel_type` | One-hot encoding |

**Why target encoding for make/model?** These columns have 50–200+ unique values.
One-hot encoding would create hundreds of sparse columns and inflate dimensionality.
Target encoding replaces each category with its mean target value, preserving
ordinality (e.g. Ferrari > BMW > Vauxhall in price) in a single feature.

In [ ]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """Mean imputer that preserves feature names for ColumnTransformer compatibility."""
    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.imputer  = SimpleImputer(strategy=self.strategy)

    def fit(self, X, y=None):
        self.imputer.fit(X)
        return self

    def transform(self, X):
        return self.imputer.transform(X)

    def get_feature_names_out(self, input_features=None):
        return input_features

def build_preprocessor() -> ColumnTransformer:
    """Build the full preprocessing pipeline. Call fit_transform on train only."""
    numerical_pipeline = Pipeline([
        ('imputer', CustomImputer(strategy='mean')),
        ('scaler',  MinMaxScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('target_encoding', ce.TargetEncoder(cols=CATEGORICAL_FEATURES)),
    ])
    one_hot_pipeline = Pipeline([
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numerical',   numerical_pipeline,   NUMERICAL_FEATURES),
        ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
        ('onehot',      one_hot_pipeline,     ONE_HOT_FEATURE),
    ])

preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_proc, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_proc,  columns=feature_names)

print(f"Processed feature matrix: {X_train_df.shape}")
print(f"Features: {list(feature_names)}")

## 4. Feature Selection — RFECV

Recursive Feature Elimination with Cross-Validation (RFECV) identifies the optimal
feature subset by iteratively removing the least important features and measuring
cross-validated R² at each step.

Applied to Linear Regression first (fast, gives a reliable signal) then to XGBoost
to compare which features each model family relies on.

In [ ]:
# RFECV with Linear Regression
rfecv_lr = Pipeline([
    ('featsel', RFECV(LinearRegression(), step=1, cv=5, scoring='r2')),
    ('regr',    LinearRegression())
])
rfecv_lr.fit(X_train_df, y_train)

n_selected_lr = rfecv_lr['featsel'].n_features_
selected_lr   = rfecv_lr['featsel'].get_feature_names_out()
print(f"RFECV selected {n_selected_lr} features for Linear Regression:")
print(list(selected_lr))

# Plot CV score vs number of features
n_scores = len(rfecv_lr['featsel'].cv_results_['mean_test_score'])
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    range(1, n_scores + 1),
    rfecv_lr['featsel'].cv_results_['mean_test_score'],
    yerr=rfecv_lr['featsel'].cv_results_['std_test_score'],
    color='steelblue', ecolor='lightblue', capsize=3
)
ax.axvline(n_selected_lr, color='red', linestyle='--', label=f'Optimal: {n_selected_lr} features')
ax.set_xlabel('Number of Features Selected')
ax.set_ylabel('CV Mean R²')
ax.set_title('RFECV — Linear Regression', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# RFECV with XGBoost — compare feature preferences between model families
rfecv_xgb = Pipeline([
    ('featsel', RFECV(xgb.XGBRegressor(random_state=RANDOM_SEED), step=1, cv=5, scoring='r2')),
    ('regr',    xgb.XGBRegressor(random_state=RANDOM_SEED))
])
rfecv_xgb.fit(X_train_df, y_train)

n_selected_xgb = rfecv_xgb['featsel'].n_features_
selected_xgb   = rfecv_xgb['featsel'].get_feature_names_out()
print(f"RFECV selected {n_selected_xgb} features for XGBoost:")
print(list(selected_xgb))

# Features selected by XGBoost but not Linear Regression
xgb_only = set(selected_xgb) - set(selected_lr)
lr_only  = set(selected_lr)  - set(selected_xgb)
print(f"\nSelected by XGBoost only: {xgb_only}")
print(f"Selected by LR only:      {lr_only}")

## 5. PCA Exploration

Principal Component Analysis is explored as an alternative dimensionality reduction
strategy. We test whether compressing the feature space into k principal components
improves model generalisation — particularly relevant for the linear models which
may suffer from multicollinearity.

**Finding:** PCA consistently reduces R² for tree-based models (Random Forest, XGBoost)
because these models handle correlated features naturally. For Linear Regression,
PCA with n=8 components recovers most of the variance with minimal performance drop.

In [ ]:
# Sweep n_components from 1 to 15 for all three model types
# This identifies the point of diminishing returns for PCA compression

pca_results = {'n_components': [], 'lr_r2': [], 'rf_r2': [], 'xgb_r2': []}

for n in range(1, 16):
    pca = PCA(n_components=n)
    X_tr_pca = pca.fit_transform(X_train_df)
    X_te_pca = pca.transform(X_test_df)

    for model_name, model in [
        ('lr',  LinearRegression()),
        ('rf',  RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED)),
        ('xgb', xgb.XGBRegressor(n_estimators=50, random_state=RANDOM_SEED)),
    ]:
        model.fit(X_tr_pca, y_train)
        pca_results[f'{model_name}_r2'].append(r2_score(y_test, model.predict(X_te_pca)))

    pca_results['n_components'].append(n)

pca_df = pd.DataFrame(pca_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pca_df['n_components'], pca_df['lr_r2'],  'b-o', label='Linear Regression')
ax.plot(pca_df['n_components'], pca_df['rf_r2'],  'g-o', label='Random Forest')
ax.plot(pca_df['n_components'], pca_df['xgb_r2'], 'r-o', label='XGBoost')
ax.set_xlabel('PCA n_components'); ax.set_ylabel('Test R²')
ax.set_title('R² vs PCA Components — All Models', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nConclusion: Tree-based models (RF, XGBoost) perform best WITHOUT PCA.")
print("Linear Regression plateaus around n=8 components.")

In [ ]:
# PCA variance explanation — understand what 8 components capture
pca_8 = PCA(n_components=8)
pca_8.fit(X_train_df)

cumulative_variance = np.cumsum(pca_8.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, 9), pca_8.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Component Variance')

axes[1].plot(range(1, 9), cumulative_variance, 'ro-')
axes[1].axhline(0.90, color='gray', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.suptitle(f'PCA (n=8) — {cumulative_variance[-1]:.1%} of variance retained',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Training & Evaluation

All models are trained on the full preprocessed feature set (no PCA) — the exploration
above confirmed this gives better R² for tree-based methods.

A shared evaluation function keeps comparisons consistent.

In [ ]:
def evaluate_model(model, X_train: pd.DataFrame, X_test: pd.DataFrame,
                   y_train: pd.Series, y_test: pd.Series, model_name: str) -> dict:
    """
    Train, evaluate, and return metrics for a single model.
    Returns a dict with r2, mae, rmse, cv_mean, cv_std.
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv   = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    print(f"{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Test R²:   {r2:.4f}")
    print(f"  MAE:       £{mae:,.0f}")
    print(f"  RMSE:      £{rmse:,.0f}")
    print(f"  CV R²:     {cv.mean():.4f} (± {cv.std():.4f})")

    return {'model': model_name, 'r2': r2, 'mae': mae, 'rmse': rmse,
            'cv_mean': cv.mean(), 'cv_std': cv.std(), 'y_pred': y_pred}

results = {}

### 6.1 Linear Regression (Baseline)

In [ ]:
lr_model = LinearRegression()
results['lr'] = evaluate_model(lr_model, X_train_df, X_test_df, y_train, y_test, 'Linear Regression')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Linear Regression — Diagnostics', fontsize=13, fontweight='bold')

y_pred_lr = results['lr']['y_pred']
axes[0].scatter(y_test, y_pred_lr, alpha=0.3, s=5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual (£)'); axes[0].set_ylabel('Predicted (£)')
axes[0].set_title(f'Actual vs Predicted (R²={results["lr"]["r2"]:.3f})')

residuals = y_test - y_pred_lr
axes[1].scatter(y_pred_lr, residuals, alpha=0.3, s=5, color='tomato')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted (£)'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

### 6.2 Random Forest

Random Forest is expected to outperform linear regression significantly here —
vehicle pricing has strong non-linear interactions (e.g. age matters more for
mass-market brands than for luxury brands).

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
results['rf'] = evaluate_model(rf_model, X_train_df, X_test_df, y_train, y_test, 'Random Forest')

# Feature importance
feature_importance = rf_model.feature_importances_
top_15_idx = np.argsort(feature_importance)[::-1][:15]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Random Forest — Results', fontsize=13, fontweight='bold')

y_pred_rf = results['rf']['y_pred']
axes[0].scatter(y_test, y_pred_rf, alpha=0.3, s=5, color='seagreen')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual (£)'); axes[0].set_ylabel('Predicted (£)')
axes[0].set_title(f'Actual vs Predicted (R²={results["rf"]["r2"]:.3f})')

axes[1].bar(range(15), feature_importance[top_15_idx], color='seagreen')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels(np.array(X_train_df.columns)[top_15_idx], rotation=90, fontsize=8)
axes[1].set_title('Top 15 Feature Importances')

plt.tight_layout()
plt.show()

### 6.3 XGBoost + Hyperparameter Tuning

XGBoost with default parameters, followed by GridSearchCV tuning.
Key parameters searched: learning rate, tree depth, subsample ratio.

In [ ]:
# XGBoost with default parameters
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', n_estimators=100,
    max_depth=5, learning_rate=0.1, random_state=RANDOM_SEED
)
results['xgb'] = evaluate_model(xgb_model, X_train_df, X_test_df, y_train, y_test, 'XGBoost')

In [ ]:
# Hyperparameter tuning via GridSearchCV
param_grid = {
    'learning_rate':    [0.1, 0.01],
    'n_estimators':     [100, 200],
    'max_depth':        [3, 5],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_SEED),
    param_grid, cv=3, scoring='neg_mean_squared_error', verbose=1, n_jobs=-1
)
grid_search.fit(X_train_df, y_train)

print(f"Best parameters: {grid_search.best_params_}")

xgb_tuned = grid_search.best_estimator_
results['xgb_tuned'] = evaluate_model(
    xgb_tuned, X_train_df, X_test_df, y_train, y_test, 'XGBoost (Tuned)'
)

### 6.4 Ensemble Voting Regressor

Combines Random Forest + Gradient Boosting + Linear Regression via soft voting
(averaging predictions). The intuition: LR captures linear relationships, RF and GB
handle non-linearities — together they partially compensate for each other's blind spots.

In [ ]:
ensemble = VotingRegressor([
    ('rf',  RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)),
    ('gb',  GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_SEED)),
    ('lr',  LinearRegression()),
])
results['ensemble'] = evaluate_model(
    ensemble, X_train_df, X_test_df, y_train, y_test, 'Ensemble Voting'
)

# Visualise individual model predictions vs ensemble on a small sample
for est_name, est in ensemble.estimators:
    est.fit(X_train_df, y_train)

sample_X = pd.concat([X_train_df, X_test_df]).head(40)
sample_y = pd.concat([y_train, y_test]).head(40)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ensemble.estimators_[0].predict(sample_X), 'g^-.', alpha=0.6, label='Random Forest')
ax.plot(ensemble.estimators_[1].predict(sample_X), 'bd:',  alpha=0.6, label='Gradient Boosting')
ax.plot(ensemble.estimators_[2].predict(sample_X), 'ys--', alpha=0.6, label='Linear Regression')
ax.plot(ensemble.predict(sample_X),                'r*-',  alpha=0.8, ms=8, label='Ensemble')
ax.plot(sample_y.values,                           'ko',   alpha=0.5, ms=4, label='Actual')
ax.set_xlabel('Sample Index'); ax.set_ylabel('Predicted Price (£)')
ax.set_title('Individual Models vs Ensemble Predictions (40 samples)', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
# Build summary table
comparison_models = ['lr', 'rf', 'xgb', 'xgb_tuned', 'ensemble']
comparison_labels = ['Linear Regression', 'Random Forest', 'XGBoost',
                     'XGBoost (Tuned)', 'Ensemble Voting']

summary_df = pd.DataFrame({
    'Model':    comparison_labels,
    'Test R²':  [round(results[m]['r2'],       4) for m in comparison_models],
    'MAE (£)':  [int(results[m]['mae'])          for m in comparison_models],
    'RMSE (£)': [int(results[m]['rmse'])         for m in comparison_models],
    'CV R²':    [round(results[m]['cv_mean'],  4) for m in comparison_models],
})
print(summary_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison', fontsize=13, fontweight='bold')

colours = ['#5B8DB8', '#E07B54', '#6BAF6B', '#9B59B6', '#F39C12']
x = np.arange(len(comparison_labels))

axes[0].bar(x, summary_df['Test R²'], color=colours, width=0.6)
axes[0].set_xticks(x); axes[0].set_xticklabels(comparison_labels, rotation=20, ha='right')
axes[0].set_ylabel('Test R²'); axes[0].set_ylim(0, 1)
axes[0].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='R²=0.8')
for i, val in enumerate(summary_df['Test R²']):
    axes[0].text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=9)
axes[0].legend()

axes[1].bar(x, summary_df['MAE (£)'], color=colours, width=0.6)
axes[1].set_xticks(x); axes[1].set_xticklabels(comparison_labels, rotation=20, ha='right')
axes[1].set_ylabel('MAE (£)')
axes[1].set_title('Mean Absolute Error — lower is better')
for i, val in enumerate(summary_df['MAE (£)']):
    axes[1].text(i, val + 100, f'£{val:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 8. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) provides model-agnostic feature importance
that shows both magnitude and direction of each feature's contribution.

Unlike Random Forest's built-in importance (which measures impurity reduction),
SHAP values represent the actual contribution to each individual prediction —
making them more reliable for understanding model behaviour.

A 5% sample is used for computational efficiency while preserving representativeness.

In [ ]:
# Sample for computational efficiency
X_train_shap = X_train_df.sample(frac=0.05, random_state=RANDOM_SEED)
y_train_shap = y_train.loc[X_train_shap.index]
X_test_shap  = X_test_df.sample(frac=0.05, random_state=RANDOM_SEED)

# Train XGBoost on sample
xgb_shap = xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_SEED)
xgb_shap.fit(X_train_shap, y_train_shap)

# Compute SHAP values
explainer   = shap.TreeExplainer(xgb_shap, X_train_shap)
shap_values = explainer(X_test_shap)

r2_shap_sample = r2_score(y_test.loc[X_test_shap.index], xgb_shap.predict(X_test_shap))
print(f"XGBoost R² on 5% sample: {r2_shap_sample:.4f} (representative of full model)")

In [ ]:
# Beeswarm plot — global feature importance with direction
# Each dot = one prediction; colour = feature value; x-position = SHAP contribution
shap.plots.beeswarm(shap_values, max_display=15,
                    show=True)

In [ ]:
# Waterfall plot — single prediction breakdown
# Shows how each feature pushed the prediction above/below the baseline
print("Explaining prediction for test sample index 0:")
shap.plots.waterfall(shap_values[0])

In [ ]:
# SHAP scatter: standard_make vs price impact
# Reveals which makes command the highest/lowest price premiums
make_idx = X_test_shap.columns.get_loc('categorical__standard_make')
shap.plots.scatter(
    shap_values[:, make_idx],
    title='SHAP Values — standard_make\n(higher x = more expensive make)'
)

In [ ]:
# SHAP scatter: standard_model coloured by body_type
# Shows whether body type moderates the model-level price effect
model_idx     = X_test_shap.columns.get_loc('categorical__standard_model')
body_type_idx = X_test_shap.columns.get_loc('categorical__body_type')
shap.plots.scatter(
    shap_values[:, model_idx],
    color=shap_values[:, body_type_idx],
    title='SHAP — standard_model (coloured by body_type)'
)

## 9. Partial Dependence Plots

PDPs show the marginal effect of each feature on the predicted price,
averaging out all other features. Useful for understanding monotonic vs
non-linear price relationships.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10), constrained_layout=True)
PartialDependenceDisplay.from_estimator(
    xgb_shap, X_test_shap,
    features=[
        'categorical__standard_make',
        'categorical__standard_model',
        'categorical__standard_colour',
        'categorical__body_type'
    ],
    kind='both',
    subsample=100,
    grid_resolution=30,
    n_jobs=2,
    random_state=RANDOM_SEED,
    ax=ax,
    n_cols=2
)
plt.suptitle('Partial Dependence Plots — Categorical Price Drivers',
             fontsize=13, fontweight='bold')
plt.show()